# Week 2 - ML Pipeline on Tesla Sales & Price Data
**Name:** Akshat Sharma

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

## Data Loading

In [ ]:
df = pd.read_csv('Data/tesla_deliveries_dataset_2015_2025.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Data Preprocessing

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month'].astype(str) + '-01')
df.head()

In [ ]:
print("Unique Regions:", df['Region'].unique())
print("Unique Models:", df['Model'].unique())
print("Unique Source Types:", df['Source_Type'].unique())

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 5))
df.groupby('Year')['Estimated_Deliveries'].sum().plot(kind='bar', color='steelblue')
plt.title('Total Deliveries by Year')
plt.xlabel('Year')
plt.ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
df['Quarter'] = df['Month'].apply(lambda x: (x - 1) // 3 + 1)
df['Delivery_Production_Ratio'] = df['Estimated_Deliveries'] / df['Production_Units']
df['Price_per_kWh'] = df['Avg_Price_USD'] / df['Battery_Capacity_kWh']
df['CO2_per_Delivery'] = df['CO2_Saved_tons'] / df['Estimated_Deliveries']
df.head()

In [ ]:
le_region = LabelEncoder()
le_model = LabelEncoder()
le_source = LabelEncoder()

df['Region_encoded'] = le_region.fit_transform(df['Region'])
df['Model_encoded'] = le_model.fit_transform(df['Model'])
df['Source_encoded'] = le_source.fit_transform(df['Source_Type'])

In [ ]:
df.head()

## Regression Modeling

In [ ]:
features = ['Year', 'Month', 'Region_encoded', 'Model_encoded', 'Production_Units',
            'Battery_Capacity_kWh', 'Range_km', 'Charging_Stations', 'Quarter',
            'Delivery_Production_Ratio', 'Price_per_kWh']

X = df[features]
y = df['Avg_Price_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    if name in ['Linear Regression', 'Ridge', 'Lasso']:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'R2': round(r2, 4)})

results_df = pd.DataFrame(results)
results_df.sort_values('R2', ascending=False)

In [ ]:
best_model_name = results_df.sort_values('R2', ascending=False).iloc[0]['Model']
print(f"Best model: {best_model_name}")

## Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(f"Best params: {grid_search.best_params_}")
print(f"Best R2 score: {grid_search.best_score_:.4f}")

In [ ]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted (Tuned Random Forest)')
plt.tight_layout()
plt.show()

In [ ]:
feat_imp = pd.Series(best_rf.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(10, 5))
feat_imp.plot(kind='barh', color='teal')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## Time Series Forecasting

In [ ]:
train_ts = ts_data[:int(len(ts_data) * 0.8)]
test_ts = ts_data[int(len(ts_data) * 0.8):]
print(f"Train size: {len(train_ts)}, Test size: {len(test_ts)}")

In [ ]:
hw_model = ExponentialSmoothing(
    train_ts,
    trend='add',
    seasonal='add',
    seasonal_periods=12
).fit()

forecast = hw_model.forecast(len(test_ts))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_ts, label='Train')
plt.plot(test_ts, label='Test')
plt.plot(forecast, label='Forecast', linestyle='--')
plt.title('Holt-Winters Forecast')
plt.xlabel('Date')
plt.ylabel('Deliveries')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
ts_rmse = np.sqrt(mean_squared_error(test_ts, forecast))
ts_mae = mean_absolute_error(test_ts, forecast)
print(f"Forecast RMSE: {ts_rmse:.2f}")
print(f"Forecast MAE: {ts_mae:.2f}")

## Conclusion

- Loaded and cleaned the Tesla deliveries dataset
- Performed EDA to understand trends across years, models and regions
- Created new features like price per kWh, delivery ratio, quarter etc.
- Trained multiple regression models to predict average price
- Tuned Random Forest using GridSearchCV for better performance
- Used Holt-Winters exponential smoothing for time series forecasting of deliveries